In [ ]:
import re
import json

In [ ]:
WIDTH = 90

# Reading raw data

In [ ]:
with open('../web/paramoji.js', 'r') as file:
    paramoji_js = file.read()

In [ ]:
m1 = re.search(r"data *= (\[([^\[\]]|\n|\[[^]]*\])*\])", paramoji_js)
m2 = m1.group(1)
m3 = re.sub(r"//", "#", m2)
data = eval(m3)
print(json.dumps(data))

In [ ]:
m1 = re.search(r"template *= (\[[^\]]*\])", paramoji_js)
m2 = m1.group(1)
m3 = re.sub(r"//", "#", m2)
template = eval(m3)
template

# Formatting data

In [ ]:
def dont_split(c):
    return c.isnumeric() or c.isalpha() or c=='.'
def formatNumbers(name, buf, w=WIDTH):
    buf = f"{name}=" + buf
    while len(buf)>0:
        v = w
        if len(buf)>v and dont_split(buf[v]):
            while dont_split(buf[v-1]):
                v = v - 1
        print(buf[0:v], end="")
        buf = buf[v:]
        if len(buf)>0:
            print("")
            buf = "    " + buf

In [ ]:
s = str(data)
s = re.sub(' ','',s)
s = re.sub(r'\[0', '[', s)
s = re.sub(r',0', ',', s)
s = re.sub(r'-0\.', '-.', s)
s = re.sub(r',+\]',']', s)
formatNumbers('  const matrix', s, 102)

In [ ]:
def trim(row):
    while len(row)>0 and row[-1]==0:
        row = row[0:-1]
    return row
trimmed_data = [trim(row) for row in data]
phpstr = "array("+ ','.join([f"array({','.join([str(x) for x in row])})" for row in trimmed_data]) + ");"
formatNumbers('$data',phpstr, w=88)

# Formatting the template

In [ ]:
def formatString(name, buf, w=WIDTH, concat_op='+', end_op=''):
    buf = f"{name}='" + buf
    while len(buf)>0:
        v = w - 2
        print(buf[0:v], end="")
        buf = buf[v:]
        if len(buf)>1:
            print("'" + concat_op)
            buf = "    '" + buf
        else:
            print(buf + "'" + end_op)
            buf = ""


In [ ]:
s = "".join([re.sub(r'^ *','',line) for line in template])
formatString('  const template', s, 102)

In [ ]:
phpstr = re.sub('<svg', '<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg"', s)
phpstr = re.sub(' href=', ' xlink:href=', phpstr)
phpstr += ""
formatString('$template', phpstr, w=88, concat_op=' .', end_op=";")

In [ ]:
index = 0
def format_data():
    global index
    row = data[index]
    index += 1
    return ",".join("%3s" % str(x) for x in row)
print("data = [")
for line in template:
    if "?" in line:
        id = re.search(r'id="([^"]*)"', line)
        if id:
            print(f"// id: {id.group(1)}")
        attrs = [a for a in re.findall(r' [a-zA-Z-]*="[^"]*"', line) if '?' in a]
        print(f"//  {' '.join(attrs)}")
        for m in re.finditer(r'\?(,\?)?', line):
            if m.group(1):
                print(f"     [{format_data()}], [{format_data()}],")
            else:
                print(f"     [{format_data()}],")
if index == len(data):
    print("  ]")